# HVG-3000 overlap check (rebuttal)

The 3,000-HVG panel was selected on all spots of each cohort, held-out donors included. `G` = that selection rerun on all donors; per LOPO fold `j`, `G_j` = the same selection (seurat_v3, batch = slide, span 0.3, raw counts) rerun on the train + val donors only, and `o_j = |G n G_j| / |G|`. No retraining.

1. **Overlap** (the answer): `o_j` per fold, and the paper rank of the genes that leave the panel.
2. **Sanity check**: the paper summaries recomputed with each held-out donor scored on (paper panel n `G_j`), against the full panel. Donor mean + Student-t 95% CI, same cascades as the paper notebooks; `diff` = paired per-donor (shared - full).

Precompute: `bash snake/scripts/canals/launch_hvg_overlap_precompute.sh` (CPU). This notebook only loads and prints.

In [1]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

PROJ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.append(str(PROJ))

OUT      = PROJ / "results" / "_analysis" / "rebuttal_hvg_overlap"
DATASETS = ["10x_TuPro", "TLS", "LUNG_CANCER"]
DATASET_LABELS = {"10x_TuPro": "Tumor Profiler", "TLS": "Kidney/Lung with TLS", "LUNG_CANCER": "NSCLC"}
SPOT_MODELS = ["homoscedastic", "heteroscedastic", "spot_homoscedastic",
               "poisson_cp", "negativebinomial_hetero_noanchor_cp"]

def ci(r):
    if not np.isfinite(r["mean"]):
        return f"{'n/a':>24s}"
    star = "*" if (r["ci_lo"] > 0 or r["ci_hi"] < 0) else " "
    return f"{r['mean']:+.4f} [{r['ci_lo']:+.4f},{r['ci_hi']:+.4f}]{star}"

# ── 1. OVERLAP ────────────────────────────────────────────────────────────────
for ds in DATASETS:
    d = OUT / ds
    if not (d / "panel.json").is_file():
        print(f"(!) {ds}: panel job not done ({d})")
        continue
    meta = json.loads((d / "panel.json").read_text())
    ov = pd.read_parquet(d / "overlap.parquet")
    print("=" * 110)
    print(f"{DATASET_LABELS[ds]}  ({meta['n_folds']} folds, scanpy {meta['scanpy_version']})")
    print(f"  all-donor rerun G vs saved paper panel (cutoff ties): symmetric difference = "
          f"{meta['n_paper_vs_all_spots_symmetric_difference']}")
    print(f"  overlap o_j: mean {meta['overlap_mean']:.4f}, min {meta['overlap_min']:.4f};  "
          f"genes of G kept in EVERY fold: {meta['n_in_all_folds']} ({meta['overlap_all_folds']:.4f})")
    print("  rank columns: scanpy highly_variable_rank (median over slides, 0-based, lower = more variable) of the genes LEAVING G (paper selection) and ENTERING G_j (fold selection)")
    print(ov[["fold", "test_donor", "n_spots_selection", "n_shared", "overlap", "n_swapped",
              "out_rank_full_median", "out_rank_full_min", "in_rank_fold_median", "in_rank_fold_min"]]
          .to_string(index=False, float_format=lambda x: f"{x:.4f}" if x < 1.5 else f"{x:.1f}"))
    print()

# ── 2. SANITY CHECK: full panel vs G n G_j ───────────────────────────────────
for ds in DATASETS:
    d = OUT / ds
    parts = [d / "scores.parquet"] + [d / f"spot_{m}.parquet" for m in SPOT_MODELS]
    missing = [p.name for p in parts if not p.is_file()]
    S = pd.concat([pd.read_parquet(p) for p in parts if p.is_file()], ignore_index=True)
    if S.empty:
        continue
    print("=" * 118)
    print(f"{DATASET_LABELS[ds]}   ('*' = donor 95% CI excludes 0)" + (f"   (!) missing: {missing}" if missing else ""))
    print(f"  {'summary':<12s} {'model':<36s} {'stat':<30s} {'full panel':>26s} {'G n G_j':>26s} {'diff (shared - full)':>26s}")
    for (summ, model, stat), g in S.groupby(["summary", "model", "stat"], sort=False):
        r = g.set_index("scope")
        print(f"  {summ:<12s} {model:<36s} {stat:<30s} {ci(r.loc['full']):>26s} "
              f"{ci(r.loc['shared']):>26s} {ci(r.loc['diff']):>26s}")
    for m in SPOT_MODELS:
        js = d / f"spot_{m}.json"
        if js.is_file():
            c = json.loads(js.read_text())["control_vs_canal2"]
            print(f"  CONTROL spot axis {m:<36s} vs canal2: {c['n_spots_matched']}/{c['n_spots_recomputed']} spots, "
                  f"max |diff| r_uD {c['max_abs_diff_r_uD']:.2e}, r_partial {c['max_abs_diff_r_partial']:.2e}, "
                  f"NaN pattern equal {c['nan_pattern_equal']}")
    print()

Tumor Profiler  (7 folds, scanpy 1.12)
  all-donor rerun G vs saved paper panel (cutoff ties): symmetric difference = 2
  overlap o_j: mean 0.9096, min 0.8547;  genes of G kept in EVERY fold: 1439 (0.4797)
  rank columns: scanpy highly_variable_rank (median over slides, 0-based, lower = more variable) of the genes LEAVING G (paper selection) and ENTERING G_j (fold selection)
 fold test_donor  n_spots_selection  n_shared  overlap  n_swapped  out_rank_full_median  out_rank_full_min  in_rank_fold_median  in_rank_fold_min
    0    MANOFYB              69083      2777   0.9257        223                1145.0              210.5               1273.0             218.0
    1    MELIPIT              68434      2781   0.9270        219                1117.0              151.5               1290.0             303.0
    2    MACEGEJ              58855      2665   0.8883        335                1219.0               48.0               1118.0             211.0
    3    MAJOFIJ              60851   